In [1]:
import argparse
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import SimpleITK as sitk
from skimage.transform import resize
from collections import OrderedDict
import warnings
warnings.filterwarnings('ignore')
from argparse import Namespace
import os
from accelerate import Accelerator
from LaMed.src.model.language_model import *
import json
from tqdm import tqdm
import monai.transforms as mtf
from generate_green_score import GenerateGreenScore
import pandas as pd
from LaMed.src.dataset.multi_dataset import prompt_templates
import re
from nltk.translate.bleu_score import sentence_bleu
from rouge import Rouge
from nltk import word_tokenize
from nltk.translate.meteor_score import meteor_score
from LaMed.src.dataset.utils import read_numpy_or_dicom
from utils.postprocessor import PostProcessor
from LaMed.src.dataset.multi_dataset import AMOSCapDataset

def clean_json_markdown(output):
    output = output.strip()
    if output.startswith("```json"):
        output = output[7:]
    if output.endswith("```"):
        output = output[:-3]
    if output.endswith("```json"):
        output = output[:-7]
    cleaned_output = output.strip()
    return cleaned_output

def fix_json_string(json_string):
    # Strip whitespace
    import re
    trimmed_string = json_string.strip()
  
    if trimmed_string.startswith('{') and not trimmed_string.startswith('[{'):
        trimmed_string = '[' + trimmed_string

    if trimmed_string.endswith('}') and not trimmed_string.endswith('}]'):
        trimmed_string = trimmed_string[:-1] + '}]'

    trimmed_string = re.sub(r',\s*([\}\]])', r'\1', trimmed_string)

    return trimmed_string

def clean_json_string(json_string):

    cleaned_string = re.sub(r'\t', 'x', json_string)
    cleaned_string = re.sub(r'[\u00d7\u2715]', 'x', cleaned_string)
    cleaned_string = re.sub(r'[\x00-\x1f\x7f]', '', cleaned_string)
    return cleaned_string

def seed_everything(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.cuda.manual_seed_all(seed)


seed_everything(42)

device = torch.device('cuda')
dtype = torch.bfloat16  # or bfloat16, float16, float32
    
model_name_or_path = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model_max_length = 3000

model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    cache_dir='/scratch/ssd004/datasets/med-img-data/amosmm/LaMed',
    torch_dtype=dtype,
    # device_map='auto',
    device_map=device,
    trust_remote_code=True,
    )

tokenizer = AutoTokenizer.from_pretrained(
    model_name_or_path,
    cache_dir='/scratch/ssd004/datasets/med-img-data/amosmm/LaMed',
    use_fast=False,
    trust_remote_code=True
)
model = model.eval()
terminators = [
    tokenizer.eos_token_id,  # End-of-sentence token
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),  # Custom end-of-conversation token
]

/fs01/home/junma/.mvlm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:46<00:00, 11.65s/it]


In [14]:
from tqdm import tqdm
import os
import json

with open("/home/jma/Documents/medvlm3d/AMOSMM_dataset.json", encoding="utf-8") as d:
  data = json.load(d)

p = "/home/jma/Documents/medvlm3d/imagesVa/"
for i in tqdm(data["validation"]):
    # try:
      name = i["image"].split("/")[-1].split(".")[0]
      findings = i["labels"]['report']["findings"]
      p_ = p + name + "/" + f"text.json"
      os.makedirs(p + name, exist_ok=True)
      # with open(p_) as f:
      #     t = json.load(f)
      # z["findings"] = t
      # z["splitted_findings"] = splitted_findings
      # if os.path.isdir(p_):
      #   os.removedirs(p_)
      with open(p_, "w") as f:
        json.dump(findings, f)
    # except:
    #     print("BEEP")

100%|██████████| 400/400 [00:00<00:00, 1868.26it/s]


In [27]:
with open("/scratch/ssd004/datasets/med-img-data/amosmm/dataset_withsplit.json", encoding="utf-8") as d:
  split = json.load(d)
wrongs = []

In [5]:
import json
with open("dataset_withsplit.json", encoding="utf-8") as d:
  data = json.load(d)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_withsplit.json'

In [21]:
from tqdm import tqdm
p = "/scratch/ssd004/datasets/med-img-data/amosmm/imagesTrProcessed/"
for i in tqdm(data["train"]):
    try:
        name = i["image"].split("/")[-1].split(".")[0]
        splitted_findings = i["labels"]["splitted_findings"]
        z = {}
        p_ = p + "/" + name + "/" + f"text.json"
        with open(p_) as f:
            t = json.load(f)
        z["findings"] = t
        z["splitted_findings"] = splitted_findings
        with open(p_, "w") as f:
            json.dump(z, f)
    except:
        print("BEEP")

 16%|█▋        | 210/1287 [00:01<00:08, 128.14it/s]

BEEP


 94%|█████████▍| 1214/1287 [00:09<00:00, 126.40it/s]

BEEP


100%|██████████| 1287/1287 [00:09<00:00, 129.78it/s]


In [19]:
for split in ["validation"]: # "val", "test"
  data_split =  data[split]
  for indx, entry in enumerate(tqdm(data_split)):

    if "splitted_findings" in entry["labels"]:
       continue
      
    # if indx not in wrongs:
    #   continue

    report = entry["labels"]["report"]
    findings = report["findings"]
    newly_generation_questions = entry.copy()
    id_ = entry['image'].split(os.sep)[-1].split(".")[0]

    keys_to_remove = []
    for k, v in findings.items():
       if len(v) <= 1:
           keys_to_remove.append(k)
    
    for key in keys_to_remove:
      findings.pop(key, '')  

    if "abdomen" not in findings.keys():
      continue 

    answer = {}

    for organ in ["abdomen"]:
      prompt = f"""
        You are an AI language model acting as an CT radiologist tasked with extracting and standardizing anatomical findings from a combined CT radiology report.
        Your goal is to split the combined report findings into specific anatomical regions detailed below and covert them into standard descriptions. Ensure each identified region is an entry from a predefined list: [liver, biliary system, spleen, pancreas, kidneys, adrenal glands, gastrointestinal tract, abdominal cavity and peritoneum, musculoskeletal system, blood vessels, lymphatic system]. If a disease or abnormality usually belongs to one of these anatomical sites, please add it to that specific site and use standard radiology report description. For sentences that describe two or more regional sites you can split them up. If you cannot assign the sentence to any anatomical region, please put it in others. Otherwise, please leave the region empty.

        IMPORTANT NOTE: If a finding references the liver surface or any other region or characteristic related to the liver, please add it under the 'liver.' If it references other organs in the biliary system apart from the liver like the gallbladder, bile duct, or hepatic portal, please add it under the 'biliary system' region. If it references the ureter, renal pelvis, or calyxes, please add it under the 'kidneys' regions. If it references the colon or intestine, please add it under the 'gastrointestinal tract' region. If it references the 'abdomen', abdominal wall, or retroperitoneum, add it under the 'abdominal cavity and peritoneum' region. If it references the aorta or inferior vena cava, add it under the 'blood vessels' region. If it references the stomach, small intestine or large intestine add it under gastrointestinal tract.

        Here is the combined report:

        <BEGIN>
        "{findings[organ]}"
        <END>

        Your output should be in JSON format with the following structure:

        JSON format: {{
            "liver": "",
            "biliary system": "",
            "spleen": "",
            "pancreas": "",
            "kidneys": "",
            "adrenal glands": "",
            "gastrointestinal tract": "",
            "abdominal cavity and peritoneum": "",
            "musculoskeletal system": "",
            "blood vessels": "",
            "lymphatic system": "",
            "others": ""
        }}

        If the report does not mention one of the areas, please leave it as an empty string. Please do not add any new sentences not mentioned in the original report.

        Only output the JSON as your answer. Your output should be JSON string that I can directly parse.
        """
      
      conversation = []
      conversation.append({"role": "user", "content": prompt})

      input_ids = tokenizer.apply_chat_template(conversation, return_tensors="pt")
      input_ids = input_ids.to(model.device)   

      generation = model.generate(input_ids, max_new_tokens=(1024 + input_ids.shape[1]), 
                                  eos_token_id=terminators,
                                  do_sample=True, top_p=0.9, temperature=1.0,
                                  pad_token_id=tokenizer.eos_token_id)[0]
      generated_text = tokenizer.decode(generation, skip_special_tokens=True)
      output_text = generated_text[len(prompt):]
      i = output_text.find("{")
      output_text = clean_json_string(clean_json_markdown(output_text[i:]))
      
      try:
        output_text = json.loads(output_text)
        answer[organ] = output_text
      except:
        print(f"Error at {indx}")
        wrongs.append(indx)
                
    data_split[indx]["labels"]["splitted_findings"] = answer

  0%|          | 0/400 [00:00<?, ?it/s]

100%|██████████| 400/400 [1:08:10<00:00, 10.23s/it]


In [25]:
tmp = data

In [29]:
split["validation"] = data

In [31]:
with open("dataset_withsplit.json", 'w') as f:
    json.dump(split, f)